In [1]:
import os
import pandas as pd
from typing import List, Tuple
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import json
from dataset_utils import split_data, TimeSeriesClassificationDataset
from torch.utils.data import DataLoader
from model.lstm import LSTMMdel
import torch
from train_utils import Trainer
import torch.nn as nn
from evaluate_utils import Evaluator

font = {'size': 16}

matplotlib.rc('font', **font)

In [2]:
DATASET_PATH = "../../dataset"
INDEX_FIELD = "timestamp"
DATA_FIELD = "num_request"
CPD_CANDIDATE_ROOT = "../../change_point_detection/offline_detection/cpd_candidate"
N_LOOKBACK = 4
N_PREDICT = 2
BATCH_SIZE = 16

LR = 1e-3
N_EPOCHS = 200
SCHEDULER_MILESTONE=[150,180]
SCHEDULER_GAMMA=0.3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
def get_data_file_list(dataset_path: str) -> List[str]:
    return os.listdir(dataset_path)

In [4]:
def read_dataset(csv_path: str,index_field:str,data_field:str) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(csv_path)
    return df[index_field].to_numpy(), df[data_field].to_numpy()

In [5]:
def read_candidate_cpds(path: str) -> List[int]:
    candidate_cpds = None
    with open(path, "r") as f:
        candidate_cpds = json.load(f)
    return candidate_cpds

In [6]:
def build_dataset(np_data: np.ndarray, candidate_cpds: List[int]):
    x = []
    y = []
    candidate_cpds = np.array(candidate_cpds, dtype=np.int32)
    np_data = np_data/20000.0
    for idx in range(len(np_data)-N_LOOKBACK-N_PREDICT+2):
        x.append(np_data[idx:idx+N_LOOKBACK+N_PREDICT-1].reshape((-1, 1)))
        future_candidate_idx = np.sum(candidate_cpds < idx+N_LOOKBACK+N_PREDICT)
        if future_candidate_idx < len(candidate_cpds):
            nearest_cpd = candidate_cpds[future_candidate_idx]
            if nearest_cpd > idx + N_LOOKBACK+N_PREDICT+N_PREDICT:
                y.append(0)
            elif nearest_cpd >= idx + N_LOOKBACK+N_PREDICT:
                y.append(1)
            # else:
            #     y.append(2)
        else:
            y.append(0)
    return x, y

In [7]:
def learn_on_dataset(x: np.ndarray, y: np.ndarray):
    x_train, x_val, x_test = split_data(x, 0.6, 0.2, 0.2)
    y_train, y_val, y_test = split_data(y, 0.6, 0.2, 0.2)
    train_dataset = TimeSeriesClassificationDataset(x_train, y_train)
    val_dataset = TimeSeriesClassificationDataset(x_val, y_val)
    test_dataset = TimeSeriesClassificationDataset(x_test, y_test)
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    model = LSTMMdel(1, 64, 2, 1).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, SCHEDULER_MILESTONE, SCHEDULER_GAMMA)
    # lr_scheduler = None
    loss_fn = nn.CrossEntropyLoss()
    trainer = Trainer(model, train_dataloader, loss_fn, optimizer, N_EPOCHS, lr_scheduler, DEVICE)
    trainer.train()
    train_evaluator = Evaluator(model, train_dataloader, loss_fn, DEVICE)
    train_evaluator.evaluate()
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    val_evaluator = Evaluator(model, val_dataloader, loss_fn, DEVICE)
    test_evaluator = Evaluator(model, test_dataloader, loss_fn, DEVICE)
    val_evaluator.evaluate()
    test_evaluator.evaluate()
    return model, train_evaluator, val_evaluator, test_evaluator

In [8]:
workload_to_skip_list = ["workload_1998-06-13", "workload_1998-06-14", "workload_1998-06-20", "workload_1998-06-21", "workload_1998-06-27", "workload_1998-06-28","workload_1998-07-04"]

In [9]:
data_file_list = get_data_file_list(DATASET_PATH)
x, y = [], []
for file_name in data_file_list:
    workload_name = file_name.split(".")[0]
    if workload_name in workload_to_skip_list:
        continue
    print("read %s" % (file_name))
    np_index, np_data = read_dataset(os.path.join(DATASET_PATH, file_name), INDEX_FIELD, DATA_FIELD)
    np_data = np.diff(np_data)
    np_data = np_data.reshape((-1, 1))
    candidate_cpds = read_candidate_cpds(os.path.join(CPD_CANDIDATE_ROOT, workload_name+".json"))
    tx, ty = build_dataset(np_data, candidate_cpds)
    x.extend(tx)
    y.extend(ty)
train_x, train_y = np.array(x), np.array(y)
model, train_evaluator, val_evaluator, test_evaluator = learn_on_dataset(train_x, train_y)

read workload_1998-06-10.csv
read workload_1998-06-11.csv
read workload_1998-06-12.csv
read workload_1998-06-15.csv
read workload_1998-06-16.csv
read workload_1998-06-17.csv
read workload_1998-06-18.csv
read workload_1998-06-19.csv
read workload_1998-06-22.csv
read workload_1998-06-23.csv
read workload_1998-06-24.csv
read workload_1998-06-25.csv
read workload_1998-06-26.csv
read workload_1998-06-29.csv
read workload_1998-06-30.csv
read workload_1998-07-03.csv
read workload_1998-07-07.csv
read workload_1998-07-08.csv
epoch: 1, loss: 0.351687
epoch: 2, loss: 0.334056
epoch: 3, loss: 0.332764
epoch: 4, loss: 0.331771
epoch: 5, loss: 0.330920
epoch: 6, loss: 0.330126
epoch: 7, loss: 0.329309
epoch: 8, loss: 0.328351
epoch: 9, loss: 0.327135
epoch: 10, loss: 0.325720
epoch: 11, loss: 0.323892
epoch: 12, loss: 0.320508
epoch: 13, loss: 0.316415
epoch: 14, loss: 0.313816
epoch: 15, loss: 0.312112
epoch: 16, loss: 0.310891
epoch: 17, loss: 0.309973
epoch: 18, loss: 0.309255
epoch: 19, loss: 0.

In [10]:
print("train accuracy: ", np.sum(np.array(train_evaluator.get_gt())==np.array(train_evaluator.get_pd()))/len(train_evaluator.get_gt()))
print("val accuracy: ", np.sum(np.array(val_evaluator.get_gt())==np.array(val_evaluator.get_pd()))/len(val_evaluator.get_gt()))
print("test accuracy: ", np.sum(np.array(test_evaluator.get_gt())==np.array(test_evaluator.get_pd()))/len(test_evaluator.get_gt()))

train accuracy:  0.9144855144855145
val accuracy:  0.8129496402877698
test accuracy:  0.7908927501497903


In [11]:
torch.save(model.state_dict(),"state_dict/lstm.pth")